## Step 1: Setting Up the Spark Environment

In a real-world company environment, we typically would not use Google Colab directly for processing large datasets. Instead, we would set up a distributed Spark environment in the cloud.

### 1. Deploy a Spark Cluster

We can deploy a **Google Cloud Dataproc cluster** to run Spark jobs.

**Cluster Configuration:**

- **Cluster Type:** Google Cloud Dataproc
- **Machine Type:** `n4-standard-4`
- **vCPUs per machine:** 4
- **Memory per machine:** 16 GB
- **Workers:** 2
- **Local Storage:** 32 GB per machine

The Dataproc cluster provides the distributed computing environment required to process large datasets using **Apache Spark**.

### 2. Store Data in Distributed Storage

Instead of keeping the dataset only on local storage, we would store it in a distributed storage system such as **HDFS**.

First, download the dataset from Kaggle:

```bash
curl -L -o ~/olist/brazilian-ecommerce.zip \
https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce

### **Key Considerations for Your Dataproc Cluster**

1.  **Cluster Resources:**

    -   **Master:** `n2-standard-4` (4 vCPUs, 16 GB RAM, 32GB disk)
    -   **Workers (2x):** `n2-standard-4` (4 vCPUs, 16 GB RAM, 64GB disk each)
    -   **Total:** 8 worker vCPUs, ~32 GB RAM (excluding master node)
2.  **Dataproc Features Disabled:**

    -   No **autoscaling**, **Metastore**, **advanced execution layer**, **advanced optimizations**
    -   **Storage:** `pd-balanced` (no SSDs, so I/O optimization is crucial)
    -   **Networking:** Internal IP **enabled**
3.  **Optimization Strategy:**

    -   Tune **shuffle partitions**, **broadcast join threshold**, and **storage persistence**
    -   Adjust **parallelism** based on **2 workers x 4 cores**
    -   Avoid **excessive caching** due to **disk-based storage**

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder\
.appName('Olist Ecommerce')\
.getOrCreate()

26/09/21 09:11:08 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [1]:
!hadoop fs -ls /user/manjurdataengineer/olist

Found 9 items
-rw-r--r--   2 manjurdataengineer hadoop    9033957 2026-09-21 07:53 /user/manjurdataengineer/olist/olist_customers_dataset.csv
-rw-r--r--   2 manjurdataengineer hadoop   61273883 2026-09-21 07:53 /user/manjurdataengineer/olist/olist_geolocation_dataset.csv
-rw-r--r--   2 manjurdataengineer hadoop   15438671 2026-09-21 07:53 /user/manjurdataengineer/olist/olist_order_items_dataset.csv
-rw-r--r--   2 manjurdataengineer hadoop    5777138 2026-09-21 07:53 /user/manjurdataengineer/olist/olist_order_payments_dataset.csv
-rw-r--r--   2 manjurdataengineer hadoop   14451670 2026-09-21 07:53 /user/manjurdataengineer/olist/olist_order_reviews_dataset.csv
-rw-r--r--   2 manjurdataengineer hadoop   17654914 2026-09-21 07:53 /user/manjurdataengineer/olist/olist_orders_dataset.csv
-rw-r--r--   2 manjurdataengineer hadoop    2379446 2026-09-21 07:53 /user/manjurdataengineer/olist/olist_products_dataset.csv
-rw-r--r--   2 manjurdataengineer hadoop     174703 2026-09-21 07:53 /user/manjur

In [2]:
# Now this hardcoded the path so we dont do from the stratch again and again
hdfs_path = '/user/manjurdataengineer/olist/'

In [3]:
customers_df = spark.read.csv(hdfs_path + 'olist_customers_dataset.csv',header = True,inferSchema = True)
geolocation_df = spark.read.csv(hdfs_path + 'olist_geolocation_dataset.csv',header = True,inferSchema = True)
order_items_df = spark.read.csv(hdfs_path + 'olist_order_items_dataset.csv',header = True,inferSchema = True)
order_payments_df = spark.read.csv(hdfs_path + 'olist_order_payments_dataset.csv',header = True,inferSchema = True)
order_reviews_df = spark.read.csv(hdfs_path + 'olist_order_reviews_dataset.csv',header = True,inferSchema = True)
orders_df = spark.read.csv(hdfs_path + 'olist_orders_dataset.csv',header = True,inferSchema = True)
products_df = spark.read.csv(hdfs_path + 'olist_products_dataset.csv',header = True,inferSchema = True)
sellers_df = spark.read.csv(hdfs_path + 'olist_sellers_dataset.csv',header = True,inferSchema = True)
category_name_translation_df = spark.read.csv(hdfs_path + 'product_category_name_translation.csv',header = True,inferSchema = True)

In [5]:
# Verifying through the printSchema All Schema data type are proper or not through the inferSchema
print("===== CUSTOMERS =====")
customers_df.printSchema()

print("===== GEOLOCATION =====")
geolocation_df.printSchema()

print("===== ORDER ITEMS =====")
order_items_df.printSchema()

print("===== ORDER PAYMENTS =====")
order_payments_df.printSchema()

print("===== ORDER REVIEWS =====")
order_reviews_df.printSchema()

print("===== ORDERS =====")
orders_df.printSchema()

print("===== PRODUCTS =====")
products_df.printSchema()

print("===== SELLERS =====")
sellers_df.printSchema()

print("===== CATEGORY NAME TRANSLATION =====")
category_name_translation_df.printSchema()

===== CUSTOMERS =====
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

===== GEOLOCATION =====
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)

===== ORDER ITEMS =====
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

===== ORDER PAYMENTS =====
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable 

In [6]:
# Data Leakage or Drop Verifying

print(f'Customers: {customers_df.count()} rows')
print(f'Geolocation: {geolocation_df.count()} rows')
print(f'Order Items: {order_items_df.count()} rows')
print(f'Order Payments: {order_payments_df.count()} rows')
print(f'Order Reviews: {order_reviews_df.count()} rows')
print(f'Orders: {orders_df.count()} rows')
print(f'Products: {products_df.count()} rows')
print(f'Sellers: {sellers_df.count()} rows')
print(f'Category Name Translation: {category_name_translation_df.count()} rows')


Customers: 99441 rows


Geolocation: 1000163 rows


Order Items: 112650 rows


Order Payments: 103886 rows


Order Reviews: 104162 rows


Orders: 99441 rows


Products: 32951 rows


Sellers: 3095 rows


Category Name Translation: 71 rows


In [7]:
# Checking the columns 
customers_df.columns

['customer_id',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state']

In [9]:
# # Now checking Null or say duplicate Value

# from pyspark.sql.functions import col

# customers_df.select([col(c).isNull().alias(c) for c in customers_df.columns]).show()

# Now checking this also for all

from pyspark.sql.functions import col, sum

dataframes = {
    "Customers": customers_df,
    "Geolocation": geolocation_df,
    "Order Items": order_items_df,
    "Order Payments": order_payments_df,
    "Order Reviews": order_reviews_df,
    "Orders": orders_df,
    "Products": products_df,
    "Sellers": sellers_df,
    "Category Translation": category_name_translation_df
}

for name, df in dataframes.items():
    print(f"\n{'='*50}")
    print(f"NULL CHECK: {name}")
    print(f"{'='*50}")
    
    df.select([
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]).show()


NULL CHECK: Customers


+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+


NULL CHECK: Geolocation


+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|                          0|              0|              0|               0|                0|
+---------------------------+---------------+---------------+----------------+-----------------+


NULL CHECK: Order Items


+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|
+--------+-------------+----------+---------+-------------------+-----+-------------+
|       0|            0|         0|        0|                  0|    0|            0|
+--------+-------------+----------+---------+-------------------+-----+-------------+


NULL CHECK: Order Payments


+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
|       0|                 0|           0|                   0|            0|
+--------+------------------+------------+--------------------+-------------+


NULL CHECK: Order Reviews


+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|        1|    2236|        2380|               92157|                 63079|                8764|                   8785|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+


NULL CHECK: Orders


+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+


NULL CHECK: Products


+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|         0|                  610|                610|                       610|               610|               2|                2|                2|               2|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+


NULL CHECK: Sellers


+---------+----------------------+-----------+------------+
|seller_id|seller_zip_code_prefix|seller_city|seller_state|
+---------+----------------------+-----------+------------+
|        0|                     0|          0|           0|
+---------+----------------------+-----------+------------+


NULL CHECK: Category Translation


+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|                    0|                            0|
+---------------------+-----------------------------+



In [13]:
# Duplicate values 
customers_df.groupBy('customer_id').count().filter('count>1').show()

# We can do like this for all the tables 

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [15]:
# Now we exploring more things in this dataset

# Customers distrubution by state

customers_df.groupBy('customer_state').count().orderBy('count',ascending=False).show()

+--------------+-----+
|customer_state|count|
+--------------+-----+
|            SP|41746|
|            RJ|12852|
|            MG|11635|
|            RS| 5466|
|            PR| 5045|
|            SC| 3637|
|            BA| 3380|
|            DF| 2140|
|            ES| 2033|
|            GO| 2020|
|            PE| 1652|
|            CE| 1336|
|            PA|  975|
|            MT|  907|
|            MA|  747|
|            MS|  715|
|            PB|  536|
|            PI|  495|
|            RN|  485|
|            AL|  413|
+--------------+-----+
only showing top 20 rows



In [16]:
orders_df.show(10)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [18]:
# Orders - order status distrubtion

orders_df.groupBy('order_status').count().orderBy('count',ascending=False).show()

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+



In [19]:
# Payments

order_payments_df.show(10)

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|        96.12|
|771ee386b001f0620...|                 1| credit_card|                   1|        81.16|
|3d7239c394a212faa...|                 1| credit_card|                   3|        51.84|
|1f78449c8

In [20]:
order_payments_df.groupBy('payment_type').count().orderBy('count',ascending=False).show()

+------------+-----+
|payment_type|count|
+------------+-----+
| credit_card|76795|
|      boleto|19784|
|     voucher| 5775|
|  debit_card| 1529|
| not_defined|    3|
+------------+-----+



In [22]:
# Top Selling Products
top_selling_products_df = order_items_df.groupBy('product_id').agg(sum('price').alias('total_sales')).orderBy('total_sales',ascending=False)

In [23]:
top_selling_products_df.show()

+--------------------+------------------+
|          product_id|       total_sales|
+--------------------+------------------+
|bb50f2e236e5eea01...|           63885.0|
|6cdd53843498f9289...| 54730.20000000005|
|d6160fb7873f18409...|48899.340000000004|
|d1c427060a0f73f6b...| 47214.51000000006|
|99a4788cb24856965...|43025.560000000085|
|3dd2a17168ec895c7...| 41082.60000000005|
|25c38557cf793876c...| 38907.32000000001|
|5f504b3a1c75b73d6...|37733.899999999994|
|53b36df67ebb7c415...| 37683.42000000001|
|aca2eb7d00ea1a7b8...| 37608.90000000007|
|e0d64dcfaa3b6db5c...|          31786.82|
|d285360f29ac7fd97...|31623.809999999983|
|7a10781637204d8d1...|           30467.5|
|f1c7f353075ce59d8...|          29997.36|
|f819f0c84a64f02d3...|29024.479999999996|
|588531f8ec37e7d5f...|28291.989999999998|
|422879e10f4668299...|26577.219999999972|
|16c4e87b98a9370a9...|           25034.0|
|5a848e4ab52fd5445...|24229.029999999962|
|a62e25e09e05e6faf...|           24051.0|
+--------------------+------------

In [25]:
# Average deleviry time analysis
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [26]:
delivery_df = orders_df.select('order_id','order_purchase_timestamp','order_delivered_customer_date')

In [27]:
delivery_df.show()

+--------------------+------------------------+-----------------------------+
|            order_id|order_purchase_timestamp|order_delivered_customer_date|
+--------------------+------------------------+-----------------------------+
|e481f51cbdc54678b...|     2017-10-02 10:56:33|          2017-10-10 21:25:13|
|53cdb2fc8bc7dce0b...|     2018-07-24 20:41:37|          2018-08-07 15:27:45|
|47770eb9100c2d0c4...|     2018-08-08 08:38:49|          2018-08-17 18:06:29|
|949d5b44dbf5de918...|     2017-11-18 19:28:06|          2017-12-02 00:28:42|
|ad21c59c0840e6cb8...|     2018-02-13 21:18:39|          2018-02-16 18:17:02|
|a4591c265e18cb1dc...|     2017-07-09 21:57:05|          2017-07-26 10:57:55|
|136cce7faa42fdb2c...|     2017-04-11 12:22:08|                         NULL|
|6514b8ad8028c9f2c...|     2017-05-16 13:10:30|          2017-05-26 12:55:51|
|76c6e866289321a7c...|     2017-01-23 18:29:09|          2017-02-02 14:08:10|
|e69bfb5eb88e0ed6a...|     2017-07-29 11:55:02|          2017-08

In [29]:
from pyspark.sql.functions import datediff

deleviry_details_df = delivery_df.withColumn('delivery_time',datediff(col('order_delivered_customer_date'),col('order_purchase_timestamp')))

In [30]:
deleviry_details_df.show()

+--------------------+------------------------+-----------------------------+-------------+
|            order_id|order_purchase_timestamp|order_delivered_customer_date|delivery_time|
+--------------------+------------------------+-----------------------------+-------------+
|e481f51cbdc54678b...|     2017-10-02 10:56:33|          2017-10-10 21:25:13|            8|
|53cdb2fc8bc7dce0b...|     2018-07-24 20:41:37|          2018-08-07 15:27:45|           14|
|47770eb9100c2d0c4...|     2018-08-08 08:38:49|          2018-08-17 18:06:29|            9|
|949d5b44dbf5de918...|     2017-11-18 19:28:06|          2017-12-02 00:28:42|           14|
|ad21c59c0840e6cb8...|     2018-02-13 21:18:39|          2018-02-16 18:17:02|            3|
|a4591c265e18cb1dc...|     2017-07-09 21:57:05|          2017-07-26 10:57:55|           17|
|136cce7faa42fdb2c...|     2017-04-11 12:22:08|                         NULL|         NULL|
|6514b8ad8028c9f2c...|     2017-05-16 13:10:30|          2017-05-26 12:55:51|   

In [31]:
deleviry_details_df.orderBy('delivery_time',ascending=False).show()

+--------------------+------------------------+-----------------------------+-------------+
|            order_id|order_purchase_timestamp|order_delivered_customer_date|delivery_time|
+--------------------+------------------------+-----------------------------+-------------+
|ca07593549f1816d2...|     2017-02-21 23:31:27|          2017-09-19 14:36:39|          210|
|1b3190b2dfa9d789e...|     2018-02-23 14:57:35|          2018-09-19 23:24:07|          208|
|440d0d17af552815d...|     2017-03-07 23:59:51|          2017-09-19 15:12:50|          196|
|2fb597c2f772eca01...|     2017-03-08 18:09:02|          2017-09-19 14:33:17|          195|
|285ab9426d6982034...|     2017-03-08 22:47:40|          2017-09-19 14:00:04|          195|
|0f4519c5f1c541dde...|     2017-03-09 13:26:57|          2017-09-19 14:38:21|          194|
|47b40429ed8cce3ae...|     2018-01-03 09:44:01|          2018-07-13 20:51:31|          191|
|2fe324febf907e3ea...|     2017-03-13 20:17:10|          2017-09-19 17:00:07|   

In [32]:
from pyspark.sql.functions import sf

deleviry_details_df.select(sf.avg('delivery_time').alias('Average Delivery Time')).show()

ImportError: cannot import name 'sf' from 'pyspark.sql.functions' (/usr/lib/spark/python/pyspark/sql/functions.py)